## Load files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

qr_df = pd.read_excel('58_full_df.xlsx', sheet_name= 'S2')

In [ ]:
# Normalize column names in both DataFrames
qr_df.columns = qr_df.columns.str.strip().str.lower()

# Confirm columns now
print(qr_df.columns)

Index(['unnamed: 0', 'question', 'medication mentioned', 'gpt-4o output',
       'llama output', 'allam output'],
      dtype='object')


In [ ]:
qr_df=qr_df.head(58)

In [ ]:
qr_df.head(2)

,unnamed: 0,question,medication mentioned,gpt-4o output,llama output,allam output
0,0,كنت اعانى من قلق واخدت دواء سبرالكس لمدة 4 شهو...,سبرالكس,"[{""Symptom"": ""قلق"", ""Diagnosis"": """", ""Treatmen...","[{""Symptom"": ""قلق"", ""Diagnosis"": """", ""Treatmen...","الإخراج:\n[{""Symptom"": ""كنت اعانى من قلق"", ""Di..."
1,1,عندي رهاب اجتماعي احس بديق النفس استعمل mediza...,"medizapin, s-citap, AVLOCARDYL","[{""Symptom"": ""ديق النفس، الافكار السلبية، النه...","[{""Symptom"": ""رهاب اجتماعي، ديق النفس، دهبة ال...","الإخراج:\n[{""Symptom"": ""رهاب اجتماعي، احس بديق..."


In [7]:
# Loading zero-shot and few-shot files to each model
import pandas as pd

def build_merged(gold_path, gpt_path, llama_path, allam_path, out_path):
    gold = pd.read_excel(gold_path, sheet_name='S2')
    gold.columns = gold.columns.str.strip().str.lower()
    gold = gold.drop(columns=[c for c in gold.columns if c.startswith('unnamed')], errors='ignore')
    gold = gold[['question', 'medication mentioned']]

    gpt4o = pd.read_excel(gpt_path)
    llama = pd.read_excel(llama_path)
    allam = pd.read_excel(allam_path)

    # اطبعي أسماء الأعمدة أول مرة تتأكدي منها
    print("GPT columns:", gpt4o.columns.tolist())
    print("LLaMA columns:", llama.columns.tolist())
    print("ALLaM columns:", allam.columns.tolist())

    gpt4o = gpt4o[['question', 'GPT-4o Output']].rename(columns={'GPT-4o Output': 'gpt-4o output'})
    llama = llama[['question', 'LLaMA Output']].rename(columns={'LLaMA Output': 'llama output'})
    allam = allam[['question', 'Allam Output']].rename(columns={'Allam Output': 'allam output'})

    merged = gold.merge(gpt4o, on='question', how='left') \
                 .merge(llama, on='question', how='left') \
                 .merge(allam, on='question', how='left')

    print(f"{out_path}: {merged.shape}")  # لازم (58, 5)
    merged.to_excel(out_path, index=False)
    return merged

# Few-shot
merged_fewshot = build_merged(
    '58_full_df.xlsx',
    'gpt4o_df.xlsx', 'llama_df.xlsx', 'allam_df.xlsx',
    '58_merged_fewshot.xlsx'
)

# Zero-shot
merged_zeroshot = build_merged(
    '58_full_df.xlsx',
    'gpt4o_df_zeroshot.xlsx', 'llama_df_zeroshot.xlsx', 'allam_df_zeroshot.xlsx',
    '58_merged_zeroshot.xlsx'
)

GPT columns: ['Unnamed: 0', 'question', 'GPT-4o Output']
LLaMA columns: ['Unnamed: 0', 'question', 'LLaMA Output']
ALLaM columns: ['question', 'Allam Output']
58_merged_fewshot.xlsx: (58, 5)
GPT columns: ['Unnamed: 0', 'question', 'GPT-4o Output']
LLaMA columns: ['Unnamed: 0', 'question', 'GPT-4o Output', 'LLaMA Output']
ALLaM columns: ['question', 'Allam Output']
58_merged_zeroshot.xlsx: (58, 5)


In [8]:
# Drop the unwanted 'unnamed: 0' column
qr_df = qr_df.drop(columns=["unnamed: 0"])

# Check result
qr_df.head()

,question,medication mentioned,gpt-4o output,llama output,allam output
0,كنت اعانى من قلق واخدت دواء سبرالكس لمدة 4 شهو...,سبرالكس,"[{""Symptom"": ""قلق"", ""Diagnosis"": """", ""Treatmen...","[{""Symptom"": ""قلق"", ""Diagnosis"": """", ""Treatmen...","الإخراج:\n[{""Symptom"": ""كنت اعانى من قلق"", ""Di..."
1,عندي رهاب اجتماعي احس بديق النفس استعمل mediza...,"medizapin, s-citap, AVLOCARDYL","[{""Symptom"": ""ديق النفس، الافكار السلبية، النه...","[{""Symptom"": ""رهاب اجتماعي، ديق النفس، دهبة ال...","الإخراج:\n[{""Symptom"": ""رهاب اجتماعي، احس بديق..."
2,السلام عليكم سبق وأن أستفسرت وأفادني أحد الأطب...,"Aripiprazole, أريبيبرازول, esciatalopram, السر...","[{""Symptom"": """", ""Diagnosis"": ""الأكتئاب، الأكت...","[{""Symptom"": """", ""Diagnosis"": ""الاكتئاب"", ""Tre...",التحليل:\n\n- Symptom: الانتكاسه، الاكتئاب، ال...
3,"اعاني من القلق , وصفلي الدكتور لوسترال ٥٠ ملج ...",لوسترال,"[{""Symptom"": ""القلق"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""القلق"", ""Diagnosis"": """", ""Treatm...","استخراج الكيانات من السؤال:\n\nالإخراج:\n[{""Sy..."
4,هل حبوب ميرزاجن لها اضرار,ميرزاجن,"[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","الإجابة:\n[{""Symptom"": ""..."", ""Diagnosis"": """",..."


In [9]:
print(pd.read_excel('gpt4o_df.xlsx').columns.tolist())
print(pd.read_excel('llama_df.xlsx').columns.tolist())
print(pd.read_excel('allam_df.xlsx').columns.tolist())

['Unnamed: 0', 'question', 'GPT-4o Output']
['Unnamed: 0', 'question', 'LLaMA Output']
['question', 'Allam Output']


## Extract Drug Entity


In [10]:
import re
from sklearn.metrics import precision_score, recall_score, f1_score

# Load the final merged file
df = qr_df.copy()

In [11]:
# ====== Fresh cleaning pipeline (explicit columns; no auto-detect) ======
import re, pandas as pd

# ---------- Splitters / patterns ----------
# split on commas/Arabic commas/semicolons and common Arabic connectors (and/or/from/to/then)
_SEP = re.compile(r'[,\u060C;؛]|\s+(?:و|او|أو|من|عن|الى|إلى|ثم)\s+', re.IGNORECASE)

_DRUG_FIELD  = re.compile(r'"Drug"\s*:\s*(?:"([^"]*)"|\[(.*?)\])', re.IGNORECASE | re.DOTALL)
_PLAIN_FIELD = re.compile(r'\bDrug\s*[:：]\s*([^\n\r]+)', re.IGNORECASE)

# Numbers (EN + Arabic-Indic) and units
_NUM   = r'[\d\u0660-\u0669]'
_UNITS = r'(?:mg|ml|mcg|ug|g|iu|ملجم|مجم|مل|قرص|أقراص|اقراص|حبة|حبه|قطرة|جرعة|أمبول|امبول|حقنة|حقن)'

# Arabic definite article (with clitics), e.g., والـ، بالـ، للـ …
_AR_ARTICLE_RE = re.compile(r'^(?:[وفبكل]\s*)?(?:ال|أل|إل|آل|لل)ـ?\s*', re.IGNORECASE)

# Leading prepositions inside tokens (e.g., "من انديرال" -> "انديرال")
_AR_PREP_PREFIX_RE = re.compile(r'^(?:من|عن|الى|إلى|على|مع|بدون|بلا)\s+', re.IGNORECASE)

# Generic dosage-form / filler words (removed anywhere)
_GENERIC_WORDS = (
    r'اسم(?:ه|ها)?|غير\s*معروف|دواء|الدواء|ادوية|الأدوية|دوا|'
    r'برشام|'
    r'قرص|أقراص|اقراص|حبة|حبه|حبوب|'
    r'كبسول(?:ة|ات)?|'
    r'شراب|'
    r'مرهم|كريم|جل|محلول|لاصقة|'
    r'بخاخ|بخّاخ|'
    r'قطرة|قطرات|'
    r'حقنة|حقن|أمبول|امبول|امبولات|'
    r'مهدئ|مسكن|منوم|'
    r'مضاد(?:\s+حيوي)?'
)
_GENERIC_INLINE_RE = re.compile(rf'\b(?:{_GENERIC_WORDS})\b', re.IGNORECASE)

# Tokens to drop completely if they remain alone
_DROP_TOKENS = {
    "و","او","أو","من","عن","على","الى","إلى","ثم","في","مع","بدون","بلا",
    "اسم","اسمه","اسمها","الدواء","دواء","دوا","برشام","حبة","حبوب",
    "قرص","اقراص","أقراص","حقنة","حقن","امبول","أمبول","شراب",
    "مرهم","كريم","جل","بخاخ","قطرة","قطرات","لبوس","تحاميل","تحميلة",
    "غير","معروف"
}
_HAS_DIGIT_RE = re.compile(r'[\d\u0660-\u0669]')

In [12]:
def _strip_noise(s: str) -> str:
    """Remove numbers/units and filler words anywhere in the token."""
    # number+unit and bare numbers anywhere (no word boundaries)
    s = re.sub(fr'{_NUM}+(?:[.,]{_NUM}+)?\s*{_UNITS}', ' ', s, flags=re.IGNORECASE)
    s = re.sub(fr'{_NUM}+(?:[.,]{_NUM}+)?', ' ', s)
    # generic/filler words
    s = _GENERIC_INLINE_RE.sub(' ', s)
    return s

In [13]:
def clean_drug_name(name: str) -> str:
    """Normalize one token until only a plausible drug name remains."""
    s = str(name).strip().lower()

    # drop article and leading prepositions inside the token
    s = _AR_ARTICLE_RE.sub('', s)
    s = _AR_PREP_PREFIX_RE.sub('', s)

    # remove inline noise (numbers+units, dosage forms, 'اسمه/اسمها', 'دواء غير معروف', etc.)
    s = _strip_noise(s)

    # keep only letters/digits/-/+; squeeze spaces and trim
    s = re.sub(r'[^\w\-\+]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip(' -+ ').strip()

    # drop if empty, still has digits, is a stop token, or is a single letter
    if (not s) or _HAS_DIGIT_RE.search(s) or (s in _DROP_TOKENS) or len(s) == 1:
        return ""
    return s

In [14]:
def extract_drugs_from_model(text) -> list[str]:
    """Pull Drug values from JSON-like or plain text, clean, and dedupe."""
    if not isinstance(text, str) or not text.strip():
        return []
    drugs = []

    # JSON-like: "Drug": "..."  or  "Drug": ["a","b"]
    for m in _DRUG_FIELD.finditer(text):
        payload = (m.group(1) if m.group(1) is not None else m.group(2)) or ""
        payload = payload.replace('"', '').replace("'", '')
        for part in _SEP.split(payload):
            part = clean_drug_name(part)
            if part:
                drugs.append(part)

    # Fallback: plain "Drug: a, b"
    for m in _PLAIN_FIELD.finditer(text):
        payload = m.group(1)
        for part in _SEP.split(payload):
            part = clean_drug_name(part)
            if part:
                drugs.append(part)

    # dedupe preserving order
    seen, out = set(), []
    for d in drugs:
        if d and d not in seen:
            seen.add(d); out.append(d)
    return out

In [15]:
def normalize_annotated(val) -> list[str]:
    """Normalize the ground-truth column to a list of clean names + dedupe."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, list):
        parts = val
    else:
        text = str(val).strip().strip('[]').replace('"', '').replace("'", '')
        parts = [p for p in _SEP.split(text) if p.strip()]

    seen, out = set(), []
    for p in parts:
        c = clean_drug_name(p)
        if c and c not in seen:
            seen.add(c); out.append(c)
    return out

In [16]:
def apply_cleaning(df: pd.DataFrame, *, med_col: str, model_cols: list[str], joiner: str = ", "):
    """Cleans in-place and returns df with strings (no lists). Requires explicit columns."""
    # drop 'unnamed' style columns (from index in CSV/Excel)
    df.drop(columns=[c for c in df.columns if c.lower().startswith("unnamed")],
            inplace=True, errors="ignore")

    # sanity checks
    missing = [c for c in [med_col, *model_cols] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    # normalize annotated -> join
    df[med_col] = df[med_col].apply(normalize_annotated).apply(lambda xs: joiner.join(xs))

    # extract from each model output -> join
    for col in model_cols:
        df[col] = df[col].apply(extract_drugs_from_model).apply(lambda xs: joiner.join(xs))

    # move medication next to question if present
    if "question" in df.columns:
        cols = list(df.columns)
        cols.insert(cols.index("question")+1, cols.pop(cols.index(med_col)))
        df = df[cols]

    return df

In [17]:
MED_COL    = "medication mentioned"
MODEL_COLS = ["gpt-4o output", "llama output", "allam output"]

In [18]:
df = apply_cleaning(df, med_col=MED_COL, model_cols=MODEL_COLS, joiner=", ")
display(df.head(58))

,question,medication mentioned,gpt-4o output,llama output,allam output
0,كنت اعانى من قلق واخدت دواء سبرالكس لمدة 4 شهو...,سبرالكس,سبرالكس,سبرالكس,سبرالكس
1,عندي رهاب اجتماعي احس بديق النفس استعمل mediza...,"medizapin, s-citap, avlocardyl","medizapin, s-citap, avlocardyl","medizapin, s-citap, avlocardyl","medizapin, s-citap, avlocardyl"
2,السلام عليكم سبق وأن أستفسرت وأفادني أحد الأطب...,"aripiprazole, أريبيبرازول, esciatalopram, سركويل","أريبيبرازول, esciatalopram, سركويل","أريبيبرازول aripiprazole, esciatalopram, سركويل","أريبيبرازول, esciatalopram, سركويل"
3,"اعاني من القلق , وصفلي الدكتور لوسترال ٥٠ ملج ...",لوسترال,لوسترال,لوسترال,لوسترال
4,هل حبوب ميرزاجن لها اضرار,ميرزاجن,ميرزاجن,ميرزاجن,ميرزاجن
5,رجاء أريد معلومات عن دواء جديد اسمه Ixel Milna...,ixel milnacipran,ixel milnacipran,ixel milnacipran,ixel milnacipran
6,انا فيني اكتئاب وقلق ووصف لي دكتورcitalopram ا...,"citalopram, ativan","citalopram, ativan","citalopram, ativan","citalopram, ativan"
7,الاعراض الانسحابية للاستالوبرام قوية علي عندما...,استالوبرام,استالوبرام,استالوبرام,استالوبرام
8,باخد اولابكس olapex و بريانيل prianil بقالي تس...,"اولابكس, بريانيل, سرترالين, مودابكس, سيرباس ol...","اولابكس, olapex, بريانيل, prianil, سرترالين, م...","اولابكس olapex, بريانيل prianil, سرترالين, مود...","اولابكس olapex, بريانيل prianil, سيرترالين mod..."
9,اخذت دواء سيروكسات ٢٥ لمدة ثلاث شهور وتركته ٥ا...,سيروكسات,سيروكسات,سيروكسات,سيروكسات


## Evaluation

In [20]:
# 1) Splitter: comma, Arabic comma "،", semicolons, and the Arabic "و" when used as "and"
_SEP = re.compile(r'[,\u060C;؛]|\s+و\s+', re.IGNORECASE)

def autodetect_cols(df: pd.DataFrame):
    """Find ground-truth and model columns by name (case-insensitive)."""
    truth_col = None
    for c in df.columns:
        lc = c.strip().lower()
        if "medication" in lc and "mention" in lc:
            truth_col = c
            break
    if truth_col is None:
        raise ValueError("Couldn't find the 'medication mentioned' column.")

    model_cols = [c for c in df.columns if "output" in c.lower()]
    if not model_cols:
        raise ValueError("Couldn't find any model output columns (names containing 'output').")
    return truth_col, model_cols

def to_set(val) -> set:
    """Turn a cell into a set of tokens with only splitting+trimming (no cleaning)."""
    if isinstance(val, list):
        parts = val
    else:
        text = "" if (val is None or (isinstance(val, float) and pd.isna(val))) else str(val)
        parts = _SEP.split(text)
    return {p.strip() for p in parts if str(p).strip()}

def evaluate(df: pd.DataFrame, truth_col=None, model_cols=None):
    """Micro-averaged P/R/F1 for each model with no additional normalization."""
    if truth_col is None or model_cols is None:
        truth_col, model_cols = autodetect_cols(df)

    y_true_sets = df[truth_col].apply(to_set)

    rows = []
    for col in model_cols:
        y_pred_sets = df[col].apply(to_set)

        TP = FP = FN = 0
        for t, p in zip(y_true_sets, y_pred_sets):
            TP += len(t & p)
            FP += len(p - t)
            FN += len(t - p)

        precision = TP / (TP + FP) if (TP + FP) else 0.0
        recall    = TP / (TP + FN) if (TP + FN) else 0.0
        f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0

        name = col.replace(" output", "").replace("Output", "").strip()
        print(f"{name}:\n  Precision: {precision:.2f}\n  Recall:    {recall:.2f}\n  F1 Score:  {f1:.2f}\n")
        rows.append((name, f1, precision, recall))

    summary = pd.DataFrame(rows, columns=["Model", "F1 Score", "Precision", "Recall"])
    summary = summary.round(2)

    return summary


summary = evaluate(df)
display(summary)

gpt-4o:
  Precision: 0.96
  Recall:    0.93
  F1 Score:  0.94

llama:
  Precision: 0.89
  Recall:    0.86
  F1 Score:  0.88

allam:
  Precision: 0.76
  Recall:    0.71
  F1 Score:  0.73



,Model,F1 Score,Precision,Recall
0,gpt-4o,0.94,0.96,0.93
1,llama,0.88,0.89,0.86
2,allam,0.73,0.76,0.71
